# bmmg candidate tables: interactive example
Load the converted tables, apply cuts, plot, and score with the trained BDT. Paths come from `config/analysis.yaml`.

In [ ]:
import sys; sys.path.insert(0, '..')
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from bmmg_ana import config, io, selection, bdt, significance, plotting
cfg = config.load_config('../config/analysis.yaml')
cfg['parquet'], cfg['channel']

In [ ]:
# what is there
for ds in cfg['samples']['data']['datasets'] + cfg['samples']['signal']['datasets']:
    print(ds, io.meta(cfg['parquet'], ds))
len(io.columns(cfg['parquet'], cfg['samples']['signal']['datasets'][0], cfg['channel']))

In [ ]:
# load a few columns of the signal MC (columns used in query/derived are added automatically)
sig = io.load(cfg['parquet'], cfg['samples']['signal']['datasets'], cfg['channel'],
              ['mass', 'vtxProb', 'mu1_pt', 'pho_pt', 'pho_mva', 'gen_B_mass'],
              query='genMatched == 1 and vtxProb > 0.01', derived={'lxySig': 'lxy / lxyErr'})
sig.describe().T

In [ ]:
# mass resolution of the signal
plt.hist(sig['mass'] - sig['gen_B_mass'], bins=100, range=(-0.6, 0.6))
plt.xlabel('m - m(gen) [GeV]');

In [ ]:
# the samples of the analysis config, and the BDT score (out-of-fold for the training samples)
s = config.load_samples(cfg, which=('signal', 'background'))
models, features = bdt.load(cfg['output'] + '/bdt')
for name, df in s.items():
    df['bdt'] = bdt.predict(models, df, features)
    plt.hist(df['bdt'], bins=50, range=(0, 1), histtype='step', density=True, label=name)
plt.yscale('log'); plt.legend(); plt.xlabel('BDT score');

In [ ]:
# cut flow with a BDT cut on top of the preselection
selection.cutflow(s['signal'], {'bdt > 0.9': 'bdt > 0.9', 'pho_pt > 3': 'pho_pt > 3'})